# Feature-selection comparison

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yazanjer/An_Explainable_AI_Education/blob/main/notebooks/03_feature_selection_comparison.ipynb)

**Answers:** Editor comment 5
**Estimated runtime:** 1 h quick / 12 h full · **Hardware:** CPU
**Quick mode:** set `QUICK_MODE = True` in the setup cell for a fast smoke test.

All selector families in the **identical** nested-CV harness, so differences are
attributable to the selector alone. BPSO is matched to VLPSO on every parameter
except the length mechanism.

---


In [ ]:
# --- Environment setup -------------------------------------------------
# Detects Colab, mounts Drive only when in Colab, installs pinned deps.
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
QUICK_MODE = True   # set False for the full budget

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT = Path("/content/drive/MyDrive/An_Explainable_AI_Education")
    PROJECT.mkdir(parents=True, exist_ok=True)
    if not (PROJECT / "src").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/yazanjer/An_Explainable_AI_Education.git", str(PROJECT)],
                       check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(PROJECT / "requirements.txt")], check=False)
else:
    PROJECT = Path(os.environ.get("VLPSO_PROJECT_ROOT", Path.cwd().parent))

os.environ["VLPSO_PROJECT_ROOT"] = str(PROJECT)
sys.path.insert(0, str(PROJECT / "src"))

from vlpso_xai.config import load_config, set_global_seeds, environment_report
cfg = load_config("quick" if QUICK_MODE else "default")
set_global_seeds(cfg.seed)
cfg.paths.mkdirs()
print("project root:", cfg.paths.root)
print("config:", cfg.config_path.name, "| hash:", cfg.hash()[:12])


In [ ]:
from vlpso_xai.selection.filters import (Chi2Filter, MutualInfoFilter,
                                         SymmetricUncertaintyFilter, ReliefFFilter)
from vlpso_xai.selection.vlpso import VLPSOSelector
from vlpso_xai.selection.bpso import BPSOSelector
from vlpso_xai.selection.base import IdentitySelector

vp = cfg.section("selection", "vlpso")
common = dict(population_size=vp["population_size"], max_iter=vp["max_iter"],
              beta_stagnation=vp["beta_stagnation"], gamma_accuracy=vp["gamma_accuracy"],
              length_penalty=vp["length_penalty"],
              interpretability_weight=vp["interpretability_weight"],
              k_neighbors=vp["k_neighbors"], random_state=cfg.seed)

SELECTORS = {
    "none": lambda: IdentitySelector(),
    "chi2": lambda: Chi2Filter(k=15),
    "mutual_info": lambda: MutualInfoFilter(k=15),
    "symmetric_uncertainty": lambda: SymmetricUncertaintyFilter(k=15),
    "relieff": lambda: ReliefFFilter(k=15),
    "bpso": lambda: BPSOSelector(max_cardinality=15, **common),
    "vlpso": lambda: VLPSOSelector(divisions=vp["divisions"], **common),
}
print(list(SELECTORS))

In [ ]:
# --- Same harness for every selector ------------------------------------
rows = []
task, (neg, pos) = "low_vs_high", ("Low", "High")
m = prim["cat_pv1"].isin([neg, pos]).to_numpy()
X = X_all[m].reset_index(drop=True)
y = (prim["cat_pv1"][m] == pos).astype(int).to_numpy()
g = prim.CNTSCHID.to_numpy()[m]

for name, factory in SELECTORS.items():
    r = run_nested_cv(X, y, g,
        models=get_models(["LogisticRegression_L2"], fast=True),
        selector_factory=factory,
        cfg=NestedCVConfig(outer_splits=3, outer_repeats=1, inner_splits=3,
                           checkpoint_dir=cfg.paths.checkpoints),
        task=task, pv=1, method=name)
    rows.append(r.assign(method=name))
sel = pd.concat(rows, ignore_index=True)
sel.to_parquet(cfg.paths.results / "selector_comparison.parquet", index=False)
display(sel.groupby("method")[["auc", "n_selected"]].agg(["mean", "std"]))